# Semantic Book Intelligence Research Lab
## Notebook 01 — Understanding Goodreads and Open Library Data

**Research question:** What entities, fields, identifiers, relationships, and quality limitations are available for the prototype?

Objectives: inspect the compressed Goodreads files safely, profile schemas and missingness, test book-author-genre relationships, evaluate ISBN coverage, and define the Open Library enrichment strategy. This notebook intentionally performs no modeling.

## Expected folder structure
```text
semantic_book_intelligence/
├── notebooks/
│   └── 01_understanding_goodreads_openlibrary.ipynb
├── data/
│   ├── raw/goodreads/
│   │   ├── goodreads_books.json.gz
│   │   ├── goodreads_book_authors.json.gz
│   │   └── goodreads_book_genres_initial.json.gz
│   ├── raw/openlibrary/api_cache/
│   ├── interim/
│   └── processed/
└── reports/
```
Keep the Goodreads files compressed; the notebook streams directly from `.gz`.

In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-Learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)

Python: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 17:06:14) [Clang 19.1.7 ]
NumPy: 2.5.1
Pandas: 3.0.3
Scikit-Learn: 1.9.0
Matplotlib: 3.11.1


In [ ]:
from pathlib import Path
import gzip, json, time
from typing import Any, Dict, Iterator, List, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
GOODREADS_DIR = DATA_DIR / "raw" / "goodreads"
OPENLIBRARY_CACHE_DIR = DATA_DIR / "raw" / "openlibrary" / "api_cache"
REPORTS_DIR = PROJECT_ROOT / "reports"
INTERIM_DIR = DATA_DIR / "interim"
for p in [OPENLIBRARY_CACHE_DIR, REPORTS_DIR, INTERIM_DIR]: p.mkdir(parents=True, exist_ok=True)

FILES = {
    "books": GOODREADS_DIR / "goodreads_books.json.gz",
    "authors": GOODREADS_DIR / "goodreads_book_authors.json.gz",
    "genres": GOODREADS_DIR / "goodreads_book_genres_initial.json.gz",
}
FILES

## 1. Verify required files

In [ ]:
def human_file_size(num_bytes: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]: return f"{size:,.2f} {unit}"
        size /= 1024

rows=[]
for name,path in FILES.items():
    rows.append({"dataset":name,"path":str(path),"exists":path.exists(),
                 "compressed_size":human_file_size(path.stat().st_size) if path.exists() else None})
file_status_df=pd.DataFrame(rows)
display(file_status_df)
missing=[str(p) for p in FILES.values() if not p.exists()]
if missing: raise FileNotFoundError("Missing required files:\n"+"\n".join(f"- {p}" for p in missing))

## 2. Streaming helpers

In [ ]:
def iter_gzip_json(path: Path) -> Iterator[Dict[str, Any]]:
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        for line_number,line in enumerate(handle,1):
            line=line.strip()
            if not line: continue
            try: yield json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path.name}, line {line_number}") from exc

def read_first_n(path: Path, n: int=5) -> pd.DataFrame:
    rows=[]
    for i,record in enumerate(iter_gzip_json(path)):
        rows.append(record)
        if i+1>=n: break
    return pd.DataFrame(rows)

def reservoir_sample(path: Path, sample_size: int, random_state: int=42) -> List[Dict[str,Any]]:
    rng=np.random.default_rng(random_state); sample=[]
    for i,record in enumerate(iter_gzip_json(path)):
        if i<sample_size: sample.append(record)
        else:
            j=int(rng.integers(0,i+1))
            if j<sample_size: sample[j]=record
    return sample

## 3. Inspect raw examples and schemas

In [ ]:
book_examples=read_first_n(FILES["books"],5)
author_examples=read_first_n(FILES["authors"],5)
genre_examples=read_first_n(FILES["genres"],5)
print("BOOKS"); display(book_examples)
print("AUTHORS"); display(author_examples)
print("GENRES"); display(genre_examples)

In [ ]:
def schema_summary(df: pd.DataFrame, name: str) -> pd.DataFrame:
    rows=[]
    for col in df.columns:
        non_null=df[col].dropna(); example=non_null.iloc[0] if len(non_null) else None
        rows.append({"dataset":name,"field":col,"dtype":str(df[col].dtype),
                     "python_type":type(example).__name__ if example is not None else None,
                     "example":repr(example)[:200] if example is not None else None})
    return pd.DataFrame(rows)

schema_df=pd.concat([schema_summary(book_examples,"books"),schema_summary(author_examples,"authors"),schema_summary(genre_examples,"genres")],ignore_index=True)
display(schema_df)

## 4. Create a manageable prototype sample
Start with 5,000 for a fast run; increase to 25,000 after validating your environment. Reservoir sampling scans the full book file to produce a uniform sample.

In [ ]:
BOOK_SAMPLE_SIZE=5_000
RANDOM_STATE=42
books_raw=pd.DataFrame(reservoir_sample(FILES["books"],BOOK_SAMPLE_SIZE,RANDOM_STATE))
print(f"Sampled {len(books_raw):,} books")
display(books_raw.head())

## 5. Normalize core fields

In [ ]:
books=books_raw.copy()
for col in ["average_rating","ratings_count","text_reviews_count","num_pages","publication_year","publication_month","publication_day"]:
    if col in books: books[col]=pd.to_numeric(books[col],errors="coerce")
for col in ["book_id","work_id"]:
    if col in books: books[col]=books[col].astype("string")
for col in ["title","description","publisher","language_code","isbn","isbn13","url","image_url"]:
    if col in books: books[col]=books[col].replace(r"^\s*$",np.nan,regex=True)
if "description" in books:
    books["has_description"]=books["description"].notna()
    books["description_length_chars"]=books["description"].fillna("").str.len()
if "authors" in books: books["author_count"]=books["authors"].apply(lambda x: len(x) if isinstance(x,list) else 0)
if "popular_shelves" in books: books["popular_shelf_count"]=books["popular_shelves"].apply(lambda x: len(x) if isinstance(x,list) else 0)
display(books.head()); books.info()

## 6. Profile missingness, uniqueness, and candidate keys

In [ ]:
def profile_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    rows=[]; total=len(df)
    for col in df.columns:
        missing=int(df[col].isna().sum())
        try: unique=int(df[col].nunique(dropna=True))
        except TypeError: unique=int(df[col].dropna().map(repr).nunique())
        rows.append({"field":col,"dtype":str(df[col].dtype),"rows":total,"missing_count":missing,
                     "missing_pct":missing/total if total else np.nan,"unique_count":unique,
                     "unique_pct":unique/total if total else np.nan})
    return pd.DataFrame(rows).sort_values(["missing_pct","unique_count"],ascending=[False,False])
books_profile=profile_dataframe(books)
display(books_profile)

checks=[]
for col in ["book_id","work_id","isbn","isbn13"]:
    if col in books:
        s=books[col].dropna()
        checks.append({"field":col,"non_null_rows":len(s),"unique_values":s.nunique(),
                       "duplicate_values":len(s)-s.nunique(),"coverage_pct":len(s)/len(books),
                       "unique_within_non_null":s.is_unique})
display(pd.DataFrame(checks))

## 7. Normalize book-author relationships and verify author coverage

In [ ]:
links=[]
for _,row in books[["book_id","authors"]].iterrows():
    if isinstance(row["authors"],list):
        for a in row["authors"]:
            if isinstance(a,dict): links.append({"book_id":str(row["book_id"]),"author_id":str(a.get("author_id")) if a.get("author_id") is not None else None,"role":a.get("role")})
book_authors=pd.DataFrame(links)
display(book_authors.head())
referenced=set(book_authors["author_id"].dropna().astype(str))
matched=[]
for record in iter_gzip_json(FILES["authors"]):
    if str(record.get("author_id")) in referenced: matched.append(record)
authors=pd.DataFrame(matched)
if "author_id" in authors: authors["author_id"]=authors["author_id"].astype("string")
author_coverage=book_authors["author_id"].isin(set(authors["author_id"])).mean() if len(book_authors) and len(authors) else np.nan
print(f"Book-author links: {len(book_authors):,}")
print(f"Matched authors: {len(authors):,}")
print(f"Relationship coverage: {author_coverage:.2%}")
display(authors.head())

## 8. Match genre records and normalize genre signals

In [ ]:
sample_book_ids=set(books["book_id"].dropna().astype(str)); matched=[]
for record in iter_gzip_json(FILES["genres"]):
    if str(record.get("book_id")) in sample_book_ids: matched.append(record)
genres_raw=pd.DataFrame(matched)
if "book_id" in genres_raw: genres_raw["book_id"]=genres_raw["book_id"].astype("string")
genre_coverage=books["book_id"].isin(set(genres_raw["book_id"])).mean() if len(genres_raw) else 0.0
rows=[]
for _,row in genres_raw.iterrows():
    if isinstance(row.get("genres"),dict):
        for genre,count in row["genres"].items(): rows.append({"book_id":row["book_id"],"genre":genre,"genre_signal_count":pd.to_numeric(count,errors="coerce")})
book_genres=pd.DataFrame(rows)
print(f"Genre-record coverage: {genre_coverage:.2%}")
if len(book_genres):
    display(book_genres.groupby("genre").agg(books=("book_id","nunique"),total_signal=("genre_signal_count","sum")).sort_values(["books","total_signal"],ascending=False).head(20))

## 9. Preliminary data-quality report

In [ ]:
quality=[]
def add(name,observed,interpretation,status): quality.append({"check":name,"observed":observed,"interpretation":interpretation,"status":status})
add("Book ID uniqueness",books["book_id"].is_unique,"Expected unique in metadata","PASS" if books["book_id"].is_unique else "REVIEW")
if "average_rating" in books:
    share=books["average_rating"].between(0,5,inclusive="both").mean(); add("Average rating range",f"{share:.2%}","Values should be within 0–5","PASS" if share>.99 else "REVIEW")
if "publication_year" in books:
    share=books["publication_year"].dropna().between(1400,2026).mean(); add("Publication year plausibility",f"{share:.2%}","Most should be 1400–2026","PASS" if share>.98 else "REVIEW")
if "has_description" in books: add("Description coverage",f"{books['has_description'].mean():.2%}","Determines NLP-ready subset","INFO")
add("Author reference coverage",f"{author_coverage:.2%}" if pd.notna(author_coverage) else "N/A","References should resolve","PASS" if pd.notna(author_coverage) and author_coverage>.99 else "REVIEW")
add("Genre record coverage",f"{genre_coverage:.2%}","Genre file is fuzzy and may be incomplete","INFO")
quality_report=pd.DataFrame(quality); display(quality_report)

## 10. Structural visualizations (not the full EDA)

In [ ]:
if "average_rating" in books:
    ax=books["average_rating"].dropna().plot(kind="hist",bins=30,title="Average Rating — Prototype Sample")
    ax.set_xlabel("Average rating"); plt.show()
if "ratings_count" in books:
    ax=np.log1p(books["ratings_count"].dropna()).plot(kind="hist",bins=40,title="Log Ratings Count — Prototype Sample")
    ax.set_xlabel("log(1 + ratings_count)"); plt.show()
if "publication_year" in books:
    years=books.loc[books["publication_year"].between(1800,2026),"publication_year"]
    ax=years.plot(kind="hist",bins=45,title="Publication Year — Prototype Sample")
    ax.set_xlabel("Publication year"); plt.show()

## 11. Open Library strategy
For Notebook 01, do **not** download the complete Open Library works and editions dumps. Instead, measure Goodreads ISBN coverage and optionally test a small number of cached ISBN lookups through the Open Library API. Set the switch below to `True` when online.

In [ ]:
coverage=[]
for col in ["isbn","isbn13"]:
    if col in books: coverage.append({"identifier":col,"available_count":int(books[col].notna().sum()),"coverage_pct":books[col].notna().mean(),"unique_count":int(books[col].nunique(dropna=True))})
display(pd.DataFrame(coverage))

RUN_OPENLIBRARY_API=False
OPENLIBRARY_SAMPLE_SIZE=20

def normalize_isbn(value: Any) -> Optional[str]:
    if pd.isna(value): return None
    value="".join(ch for ch in str(value) if ch.isdigit() or ch.upper()=="X")
    return value or None

def fetch_openlibrary_book_by_isbn(isbn: str) -> Dict[str,Any]:
    import requests
    isbn=normalize_isbn(isbn)
    if not isbn: return {}
    cache_path=OPENLIBRARY_CACHE_DIR/f"ISBN_{isbn}.json"
    if cache_path.exists(): return json.loads(cache_path.read_text(encoding="utf-8"))
    response=requests.get(f"https://openlibrary.org/isbn/{isbn}.json",timeout=30,headers={"User-Agent":"SemanticBookIntelligencePrototype/0.1"})
    payload={} if response.status_code==404 else response.json()
    if response.status_code not in (200,404): response.raise_for_status()
    cache_path.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding="utf-8")
    return payload

if RUN_OPENLIBRARY_API:
    results=[]
    candidates=books.loc[books["isbn13"].notna(),["book_id","title","isbn13"]].drop_duplicates("isbn13").head(OPENLIBRARY_SAMPLE_SIZE)
    for _,row in candidates.iterrows():
        p=fetch_openlibrary_book_by_isbn(row["isbn13"])
        results.append({"goodreads_book_id":row["book_id"],"goodreads_title":row["title"],"isbn13":row["isbn13"],"openlibrary_found":bool(p),"openlibrary_key":p.get("key"),"openlibrary_title":p.get("title"),"publish_date":p.get("publish_date"),"publisher_count":len(p.get("publishers",[])),"subject_count":len(p.get("subjects",[])),"work_count":len(p.get("works",[]))})
        time.sleep(.25)
    openlibrary_matches=pd.DataFrame(results); display(openlibrary_matches)
else: print("Open Library API test disabled. Set RUN_OPENLIBRARY_API=True to run it.")

## 12. Export prototype tables and reports

In [ ]:
try:
    books.drop(columns=["authors","popular_shelves"],errors="ignore").to_parquet(INTERIM_DIR/"books_sample.parquet",index=False)
    authors.to_parquet(INTERIM_DIR/"authors_sample.parquet",index=False)
    book_authors.to_parquet(INTERIM_DIR/"book_authors_sample.parquet",index=False)
    genres_raw.to_parquet(INTERIM_DIR/"genres_sample_raw.parquet",index=False)
    book_genres.to_parquet(INTERIM_DIR/"book_genres_sample.parquet",index=False)
    print("Parquet exports complete")
except ImportError:
    print("Install pyarrow for Parquet export: pip install pyarrow")
books_profile.to_csv(REPORTS_DIR/"notebook01_books_field_profile.csv",index=False)
quality_report.to_csv(REPORTS_DIR/"notebook01_data_quality_report.csv",index=False)
schema_df.to_csv(REPORTS_DIR/"notebook01_schema_examples.csv",index=False)
print(f"Reports written to {REPORTS_DIR}")

## 13. Conclusions to complete after execution
- Book metadata coverage:
- Description coverage:
- ISBN/ISBN-13 coverage:
- Author relationship integrity:
- Genre coverage and limitations:
- Fields requiring cleaning:
- Fields suitable for Open Library enrichment:

### Decisions for Notebook 02
- Prototype sample size:
- Inclusion/exclusion rules:
- Core EDA variables:
- Outlier policies to investigate:
- Open Library API versus local dump strategy:

### Implications for Lane 1
Record which schema choices, identifiers, quality checks, and ingestion methods appear production-worthy.